# Plant Disease Classification – Data Preprocessing

In [20]:
from torchvision import datasets

DATASET_DIR = "datasets/plantvillage dataset/color"

full_dataset = datasets.ImageFolder(root=DATASET_DIR)

class_names = full_dataset.classes
num_classes = len(class_names)

print("Total images:", len(full_dataset))
print("Number of classes:", num_classes)

Total images: 54305
Number of classes: 38


## DEFINE TRANSFORMS



In [21]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## TRAIN VALIDATION SPLIT

In [22]:
from torch.utils.data import random_split

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size]
)

# Assign transforms AFTER split
train_dataset.dataset.transform = train_transforms
val_dataset.dataset.transform = val_transforms

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

Train samples: 43444
Val samples: 10861


## HANDLE CLASS IMBALANCE

In [23]:
import numpy as np
from torch.utils.data import WeightedRandomSampler

# Extract labels ONLY from train subset
train_indices = train_dataset.indices
train_labels = [full_dataset.samples[i][1] for i in train_indices]

class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts

sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

## CREATE TRAIN DATALOADER

In [24]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created successfully")

DataLoaders created successfully


In [26]:
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

torch.Size([32, 3, 224, 224]) torch.Size([32])
